# Whisper Pro: Procesamiento Masivo y Recursivo

Este cuaderno está configurado para escanear recursivamente el dataset en `/kaggle/input/datasets/danieldobles/test-whisper`, transcribir todos los audios encontrados usando **Whisper large-v3** y guardar los resultados en `/kaggle/working/transcriptions/`.

In [ ]:
# 1. Instalación de dependencias
!pip install -U openai-whisper
!apt-get install -y ffmpeg

In [ ]:
import whisper
import torch
import os
import datetime
from pathlib import Path

# --- CONFIGURACIÓN ---
INPUT_DIR = "/kaggle/input/datasets/danieldobles/test-whisper"
OUTPUT_DIR = "/kaggle/working/transcriptions"
MODEL_NAME = "large-v3"
AUDIO_EXTENSIONS = (".mp3", ".wav", ".m4a", ".flac", ".ogg", ".mp4")

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

# --- CARGA DEL MODELO ---
print(f"Cargando modelo {MODEL_NAME}...")
model = whisper.load_model(MODEL_NAME, device=device)
print("Modelo cargado correctamente.")

In [ ]:
def transcribe_file(file_path):
    """Transcribe un archivo individual y guarda los resultados."""
    file_path = Path(file_path)
    output_base = Path(OUTPUT_DIR) / file_path.stem
    
    # Inteligencia: Omitir si ya existe la transcripción
    if os.path.exists(f"{output_base}.txt"):
        print(f"[OMITIDO] Ya existe transcripción para: {file_path.name}")
        return

    print(f"\n[PROCESANDO] {file_path}")
    start_time = datetime.datetime.now()
    
    try:
        result = model.transcribe(
            str(file_path),
            verbose=False,  # Cambiar a True para ver segmentos en tiempo real
            beam_size=5,
            fp16=True
        )
        
        # Guardar en TXT
        with open(f"{output_base}.txt", "w", encoding="utf-8") as f:
            f.write(result["text"])
        
        # Guardar en MD
        with open(f"{output_base}.md", "w", encoding="utf-8") as f:
            f.write(f"# Transcripción de {file_path.name}\n\n{result['text']}")
            
        duration = datetime.datetime.now() - start_time
        print(f"[OK] Completado en {duration}. Guardado en {OUTPUT_DIR}")
        
    except Exception as e:
        print(f"[ERROR] Falló la transcripción de {file_path.name}: {e}")

# --- BUCLE PRINCIPAL RECURSIVO ---
print(f"Escaneando archivos en {INPUT_DIR}...")
files_to_process = []
for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower().endswith(AUDIO_EXTENSIONS):
            files_to_process.append(os.path.join(root, file))

print(f"Encontrados {len(files_to_process)} archivos de audio.")

for audio_file in files_to_process:
    transcribe_file(audio_file)

print("\n--- PROCESO FINALIZADO ---")